# EasyVVUQ with OSS-DBS: three-parameter impedance demo

This notebook runs a small local uncertainty quantification workflow around OSS-DBS. It uses EasyVVUQ to sample three Cole-Cole encapsulation-layer parameters, calls OSS-DBS for each sample, and analyzes the impedance spectra with PCE and stochastic collocation.

The runner uses `input_files/input_cc.json` from OSS-DBS as the base configuration and writes one `output.csv` per EasyVVUQ run.

In [ ]:
import sys
import time
from pathlib import Path

from scipy.stats import norm

try:
    import chaospy as cp
    import easyvvuq as uq
    import matplotlib.pyplot as plt
    import numpy as np
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Install the notebook dependencies in this OSS-DBS environment: "
        "python -m pip install easyvvuq chaospy pandas matplotlib"
    ) from exc

EXAMPLE_DIR = Path.cwd()
RUNNER = EXAMPLE_DIR / "uq_ossdbs_runner.py"
TEMPLATE = EXAMPLE_DIR / "easyvvuq_input.template"
OUTPUT_DIR = EXAMPLE_DIR / "easyvvuq_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

QOI_COLS = ["real", "imag"]
PARAMETER_NAMES = ["sigma", "alpha_3", "eps_delta_3"]
print("Example directory:", EXAMPLE_DIR)
print("Python:", sys.executable)

## Parameter Distributions

The ranges below are intentionally broad enough to make the surrogate response visible, but small enough for a quick local example. Adjust them for a scientific study.

In [ ]:
def vary_three_parameters():
    """Return uncertain parameter distributions."""
    return {
        "sigma": cp.Uniform(0.3, 1.2),
        "alpha_3": cp.Uniform(0.0, 0.4),
        "eps_delta_3": cp.Uniform(0.0, 8.0e4),
    }


vary_three_parameters()

In [ ]:
def create_campaign(name, work_dir):
    """Create an EasyVVUQ campaign for OSS-DBS runs."""
    work_dir.mkdir(parents=True, exist_ok=True)
    encoder = uq.encoders.GenericEncoder(
        template_fname=str(TEMPLATE),
        delimiter="$",
        target_filename="input.json",
    )
    decoder = uq.decoders.SimpleCSV(
        target_filename="output.csv",
        output_columns=QOI_COLS,
    )
    actions = uq.actions.Actions(
        uq.actions.CreateRunDirectory(str(work_dir), flatten=True),
        uq.actions.Encode(encoder),
        uq.actions.ExecuteLocal(
            f"{sys.executable} {RUNNER} input.json",
            stdout="stdout.txt",
            stderr="stderr.txt",
        ),
        uq.actions.Decode(decoder),
    )
    return uq.Campaign(
        name=name,
        work_dir=str(work_dir),
        params={
            "sigma": {"type": "float", "default": 0.7},
            "alpha_3": {"type": "float", "default": 0.0},
            "eps_delta_3": {"type": "float", "default": 0.0},
            "freq": {"type": "float", "default": 0.0},
            "real": {"type": "float", "default": 0.0},
            "imag": {"type": "float", "default": 0.0},
        },
        actions=actions,
    )

## Polynomial Chaos Expansion

In [ ]:
PCE_ORDER = 2

pce_campaign = create_campaign(
    "ossdbs_three_parameter_pce", OUTPUT_DIR / "pce_campaign"
)
pce_sampler = uq.sampling.PCESampler(
    vary=vary_three_parameters(),
    polynomial_order=PCE_ORDER,
    regression=True,
)

start = time.perf_counter()
pce_campaign.set_sampler(pce_sampler)
pce_campaign.execute().collate(progress_bar=True)
pce_execution_seconds = time.perf_counter() - start

pce_analysis = uq.analysis.PCEAnalysis(sampler=pce_sampler, qoi_cols=QOI_COLS)
pce_campaign.apply_analysis(pce_analysis)
pce_results = pce_campaign.get_last_analysis()
print(f"PCE completed in {pce_execution_seconds:.1f} s")

## Stochastic Collocation

In [ ]:
SC_SPARSE = True
SC_POLYNOMIAL_ORDER = 2

sc_campaign = create_campaign("ossdbs_three_parameter_sc", OUTPUT_DIR / "sc_campaign")
sc_sampler = uq.sampling.SCSampler(
    vary=vary_three_parameters(),
    polynomial_order=SC_POLYNOMIAL_ORDER,
    sparse=SC_SPARSE,
)

start = time.perf_counter()
sc_campaign.set_sampler(sc_sampler)
sc_campaign.execute().collate(progress_bar=True)
sc_execution_seconds = time.perf_counter() - start

sc_analysis = uq.analysis.SCAnalysis(sampler=sc_sampler, qoi_cols=QOI_COLS)
sc_campaign.apply_analysis(sc_analysis)
sc_results = sc_campaign.get_last_analysis()
print(f"SC completed in {sc_execution_seconds:.1f} s")

## Mean and 90% Prediction Interval

EasyVVUQ gives mean and standard deviation for each output column. Here we use a normal approximation for a compact visual summary.

In [ ]:
for stat in ["mean", "std", "p10", "p90", "percentile_5", "percentile_95"]:
    try:
        print(stat, np.asarray(pce_results.describe(QOI_COLS[0], stat)).shape)
    except Exception as e:
        print(stat, "not available:", e)

In [ ]:
FREQUENCIES = np.array(
    [
        3000.0,
        6207.41424334437,
        12843.997196158187,
        700716.4407270363,
        3000000.0,
    ]
)


def plot_prediction_interval(results, qoi, label, interval=0.9):
    """Plot the mean and normal-approximation interval for one QoI."""
    mean = np.asarray(results.describe(qoi, "mean"), dtype=float)
    std = np.asarray(results.describe(qoi, "std"), dtype=float)
    z = norm.ppf(1.0 - (1.0 - interval) / 2.0)

    lower = mean - z * std
    upper = mean + z * std
    plt.plot(FREQUENCIES, mean, marker="o", label=f"{label} mean")
    plt.fill_between(
        FREQUENCIES,
        lower,
        upper,
        alpha=0.2,
        label=f"{label} {int(interval * 100)}% normal interval",
    )


for qoi in QOI_COLS:
    plt.figure(figsize=(7, 4))
    plot_prediction_interval(pce_results, qoi, "PCE")
    plot_prediction_interval(sc_results, qoi, "SC")
    plt.xscale("log")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel(f"Z_{qoi}")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.show()

## First-Order Sobol Indices

In [ ]:
def plot_first_sobols(results, qoi, title):
    """Plot first-order Sobol indices for one QoI."""
    plt.figure(figsize=(7, 4))
    for parameter in PARAMETER_NAMES:
        values = np.asarray(results.sobols_first(qoi, parameter), dtype=float)
        plt.plot(FREQUENCIES, values, marker="o", label=parameter)
    plt.xscale("log")
    plt.ylim(bottom=0)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("First-order Sobol index")
    plt.title(title)
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.show()


for qoi in QOI_COLS:
    plot_first_sobols(pce_results, qoi, f"PCE Sobol indices for Z_{qoi}")
    plot_first_sobols(sc_results, qoi, f"SC Sobol indices for Z_{qoi}")